# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Artasam/Machine-Learning/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane because the starter pipeline (`scripts/01–05`) already builds a working,
transparent baseline score and reports `Precision@50` for it. That gives me a real pipeline to
learn from and try to beat, instead of starting from a blank page. It also matches a real
operational problem: a content team can't manually review 30,000 pages a week, so they need a
short, defensible list of which pages to look at first — a ranking problem, which is exactly
what this lane is built for.

This is provisional — I can confirm or change it until the end of Week 4. Lane 1 (Ranking
Signal Analysis) is my backup if the decline label I build in Week 3 turns out too noisy under
client-holdout validation.

The check below confirms the columns this lane depends on actually exist and are usable in the
starter file before I commit more time to it.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

lane2_cols = ["trend_direction", "days_since_last_update", "ctr", "avg_position", "impressions_90d"]
print("Rows, cols:", df.shape)
print("Lane 2 columns present:", all(c in df.columns for c in lane2_cols))
print()
print(df[lane2_cols].isna().mean().round(3).rename("pct_missing"))

Rows, cols: (30000, 44)
Lane 2 columns present: True

trend_direction           0.0
days_since_last_update    0.0
ctr                       0.0
avg_position              0.0
impressions_90d           0.0
Name: pct_missing, dtype: float64


## 2. The question: decision, action, cost of a wrong call

**Search question:** Given a page's 90-day search, content, and engagement signals, which
pages should a content reviewer look at first this week?

**Unit of analysis:** one **page** (`content_id`), described by its latest 90-day window,
belonging to one **client** (`client_id`).

**Output:** a ranked review queue — score, a short reason code (declining with demand, stale
and visible, low CTR for its position), and a suggested action (refresh, expand, protect,
prune, monitor).

**The action someone takes:** a content strategist works down the queue and spends limited
weekly review time — realistically the top 20–50 pages — on the highest-ranked candidates
first, instead of reviewing at random.

**Cost of a wrong call:**
- *False positive* — a flagged page that didn't need attention: wastes scarce reviewer time.
- *False negative* — a genuinely declining page that never surfaces: the more expensive
  failure, since it can keep losing traffic unnoticed for weeks.

**Why data/ML, not just a rule:** a hand-written rule already exists in this repo
(`02_baseline_score.py`). The real question isn't "should we use ML" — it's whether a learned
ranking beats that rule by enough to justify the added complexity. The check below sizes the
problem: how many pages would a fixed capacity of 50 reviews/week miss if the team just went
in random order.

In [2]:
weekly_capacity = 50
qualifying = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]

print(f"Pages needing review by this definition: {len(qualifying):,} of {len(df):,}")
print(f"Weekly review capacity: {weekly_capacity}")
print(f"Weeks to clear the backlog at this capacity: {len(qualifying) / weekly_capacity:.0f}")
print(f"If reviewed in RANDOM order instead of ranked, "
      f"expected pages missed in week 1 alone: {len(qualifying) - weekly_capacity:,}")

Pages needing review by this definition: 13,152 of 30,000
Weekly review capacity: 50
Weeks to clear the backlog at this capacity: 263
If reviewed in RANDOM order instead of ranked, expected pages missed in week 1 alone: 13,102


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter CSV (`data/raw/content_refresh_anonymized.csv`, 30,000 pages,
44 columns), checking whether this lane has enough signal and volume to be worth 7 weeks.

In [3]:
# Number 1: how many pages match a simple "declining with real demand" definition
declining_with_demand = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
n1, pct1 = declining_with_demand.sum(), declining_with_demand.mean() * 100
print(f"1) declining_with_demand pages: {n1:,} of {len(df):,} ({pct1:.1f}%)")

# Number 2: how many visible pages are under-capturing clicks for their position
low_ctr_visible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)
n2, pct2 = low_ctr_visible.sum(), low_ctr_visible.mean() * 100
print(f"2) low_ctr_visible_page pages: {n2:,} ({pct2:.1f}%)")

# Number 3: breadth of the dataset (clients, typical scale) — needed for client-holdout validation
print(f"3) Unique clients: {df['client_id'].nunique()}  |  "
      f"Median impressions_90d: {df['impressions_90d'].median():.0f}  |  "
      f"Median content_age_days: {df['content_age_days'].median():.0f}")

1) declining_with_demand pages: 13,152 of 30,000 (43.8%)
2) low_ctr_visible_page pages: 9,759 (32.5%)
3) Unique clients: 32  |  Median impressions_90d: 731  |  Median content_age_days: 236


**Reading these:** ~13,150 pages (44%) qualify as declining-with-demand — a real pool, not
a trickle and not almost everything, so ranking them is a genuine prioritization problem. A
separate 32.5% of visible pages show a CTR gap for their position, so the queue has more than
one independent reason code, which keeps it from collapsing into one repetitive signal. 32
distinct clients means I can hold out entire clients for validation later, the same way the
starter pipeline does, instead of splitting rows randomly and letting a model memorize a
client's overall traffic level.

For reference, the starter pipeline's own `outputs/model_report.md` (from running the full
`scripts/run_all.py`, not reproduced in this notebook) reports Precision@50 of about 0.240 for
the baseline rule versus about 0.740 for a random forest — a roughly 3x lift, which is evidence
there's real signal beyond the hand-written formula, not a promise about my own model's
eventual score.

## 4. Careful words: what I can and can't claim

**Can claim:** observed, historical patterns between search/engagement signals and a *proxy*
label (`trend_direction == "down"`) measured in the current window; a ranked queue for
*human review* — decision support, not an autonomous decision; a comparison of a learned model
against a transparent baseline rule on clients the model never trained on.

**Cannot claim:** that any feature is a Google ranking factor, or that refreshing a page
*causes* recovery — nothing here is a controlled experiment. I also can't yet claim the
starter's `trend_direction` label is the right target; it's computed from the *current* window,
not a *future* outcome, so a stronger version would need a forward-looking label with a real
leakage check.

The check below is exactly that leakage risk made concrete: how tangled the "current-window"
trend label is with the raw current-window metrics I'd also use as features, which is why I
can't yet call the model's future output "predictive" without redoing this properly.

In [4]:
# Simple correlation sanity check: current-window features vs. the current-window label.
# High correlation here is EXPECTED (both are measuring the same 90-day window) and is a
# warning sign, not a result -- it shows why "down this window" cannot be swapped for
# "will decline next window" without a proper time-based split.
trend_numeric = df["trend_pct"]
corr_with_ctr = trend_numeric.corr(df["ctr"])
corr_with_position = trend_numeric.corr(df["avg_position"])

print(f"corr(trend_pct, ctr):          {corr_with_ctr:.3f}")
print(f"corr(trend_pct, avg_position): {corr_with_position:.3f}")
print("Both describe the SAME 90-day window as the label -- not evidence of forecasting power.")

corr(trend_pct, ctr):          0.008
corr(trend_pct, avg_position): 0.047
Both describe the SAME 90-day window as the label -- not evidence of forecasting power.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (all IDs are the repo's pseudonymized hashes)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.